# Qwen3-8B local-GPU smoke test

This notebook first runs three rows from condition 0 on one GPU batch at a time. Once the smoke test succeeds, change `TARGET_CONDITIONS` and `NUM_ROWS` in the settings cell for the full experiment. Existing successful rows are resumed automatically.

In [ ]:
from pathlib import Path
import importlib
import sys

def find_repo_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / 'content conditions').is_dir() and (candidate / 'gpu_experiments').is_dir():
            return candidate
    raise FileNotFoundError('Could not locate the project root')

REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import gpu_experiments.model_loader
import gpu_experiments.inference
import gpu_experiments.pipeline
importlib.reload(gpu_experiments.model_loader)
importlib.reload(gpu_experiments.inference)
importlib.reload(gpu_experiments.pipeline)
from gpu_experiments.model_loader import ModelConfig
from gpu_experiments.pipeline import run_pipeline
print('Project root:', REPO_ROOT)

## GPU check

In [ ]:
import torch
import transformers

if not torch.cuda.is_available():
    raise RuntimeError('PyTorch cannot see a CUDA GPU. Check the PyTorch/CUDA installation.')

free_bytes, total_bytes = torch.cuda.mem_get_info(0)
print('PyTorch:', torch.__version__)
print('Transformers:', transformers.__version__)
print('GPU:', torch.cuda.get_device_name(0))
print('Compute capability:', torch.cuda.get_device_capability(0))
print(f'VRAM free/total: {free_bytes / 2**30:.2f}/{total_bytes / 2**30:.2f} GiB')

major, minor = torch.cuda.get_device_capability(0)
if (major, minor) < (7, 5):
    raise RuntimeError('This smoke-test configuration expects compute capability 7.5 or newer.')

## Safe smoke-test settings

In [ ]:
MODEL = ModelConfig(
    model_id='Qwen/Qwen3-8B',
    dtype='float16',
    device_map='auto',
    attention_implementation='sdpa',
)

TARGET_CONDITIONS = [0]  # Any subset of 0..8, 20, and 21
NUM_ROWS = 3
START_ROW = 0
MAX_CONCURRENCY = 1
MAX_NEW_TOKENS = 2048
TEMPERATURE = 0.0
ENABLE_THINKING = False

## Run or resume

The first run downloads and loads the model, so it will take longer. Results are flushed to disk after every completed batch.

In [ ]:
result = run_pipeline(
    teacher_model='deepseek-v4-flash',
    model=MODEL,
    condition=TARGET_CONDITIONS,
    num_rows=NUM_ROWS,
    max_concurrency=MAX_CONCURRENCY,
    start_row=START_ROW,
    temperature=TEMPERATURE,
    max_new_tokens=MAX_NEW_TOKENS,
    enable_thinking=ENABLE_THINKING,
    retry_failed=True,
    repo_root=REPO_ROOT,
)
result

## After the smoke test

Keep `MAX_CONCURRENCY = 1` for the first full condition. Increase it to `2` only after confirming stable VRAM usage. Set `NUM_ROWS = None` only when you are ready to process every aligned row.